# Asymptotic Limit Computation

This notebook demonstrates the difference between:
1. **Numerical parameters** (Python floats) - limits compute instantly
2. **Symbolic parameters** (sympy symbols) - limits can hang

In [ ]:
from sympy import symbols, Abs, limit, N, oo, Pow
import numpy as np

## 1. Numerical Parameters (Fast)

When `a0` and `a1` are Python floats, sympy immediately simplifies the expression.
Notice that `Abs(-3.83 - x)` becomes `(x + 3.83)` because sympy knows the sign.

In [14]:
x = symbols('x', positive=True)

# These are Python floats, NOT sympy symbols
# a0 = -3.83
# a1 = -2.81

a0 = -53.51139
a1 = -2.647826

expr_numerical = Pow(Abs(a0 - x), a1)
print(f"Expression: Pow(Abs({a0} - x), {a1})")
print(f"Sympy simplifies to: {expr_numerical}")
print(f"\nNotice: Abs() is gone! Sympy knew that {a0} - x < 0 for positive x")

Expression: Pow(Abs(-53.51139 - x), -2.647826)
Sympy simplifies to: (x + 53.51139)**(-2.647826)

Notice: Abs() is gone! Sympy knew that -53.51139 - x < 0 for positive x


In [15]:
# Limits compute instantly
print(f"lim(x -> inf) = {limit(expr_numerical, x, oo)}")
print(f"lim(x -> 0+) = {limit(expr_numerical, x, 0, '+')}")

lim(x -> inf) = 0
lim(x -> 0+) = 10000000000000*10**(23913/100000)*3**(176087/250000)*594571**(176087/500000)/153228198905979698619


## 2. Symbolic Parameters (Can Hang!)

When `a0` and `a1` are sympy symbols, sympy doesn't know:
- Is `a0 - x` positive or negative?
- Is `a1` positive or negative?

This causes sympy to explore many cases and can hang indefinitely.

In [ ]:
# Now a0, a1 are SYMBOLS (unknown values)
a0_sym, a1_sym = symbols('a0 a1', real=True)

expr_symbolic = Pow(Abs(a0_sym - x), a1_sym)
print(f"Expression: {expr_symbolic}")
print(f"\nNotice: Abs() is still there because sympy doesn't know the sign of (a0 - x)")

In [ ]:
# WARNING: This cell may HANG! Uncomment to test (Ctrl+C to interrupt)
# print("Computing limit with symbolic parameters...")
# result = limit(expr_symbolic, x, oo)
# print(f"Result: {result}")

## 3. The Solution: Substitute Then Compute

Start with symbolic expression, substitute numerical values, then compute limit.

In [ ]:
# Start with symbolic expression
print(f"Symbolic: {expr_symbolic}")

# Substitute numerical values
a0_val = -3.83
a1_val = -2.81
expr_substituted = expr_symbolic.subs({a0_sym: a0_val, a1_sym: a1_val})
print(f"After .subs(): {expr_substituted}")

# Now limits work
print(f"\nlim(x -> inf) = {limit(expr_substituted, x, oo)}")
print(f"lim(x -> 0+) = {limit(expr_substituted, x, 0, '+')}")

## 4. Why Does This Matter?

In the fitting code, expressions are parsed from strings like `"pow(Abs(a0 - x), a1)"`.
The parser creates sympy symbols for `a0`, `a1`, so the expression is symbolic.

When checking asymptotes **before** fitting, we don't have numerical values yet,
so sympy may hang trying to compute limits.

When checking asymptotes **after** fitting, we have numerical parameter values
and should substitute them before computing limits.

In [ ]:
# Simulate what the parser does
from sympy import sympify, Lambda, sqrt

def parse_expression(expr_str):
    """Parse expression string like the SympyParser does."""
    x = symbols('x', positive=True)
    a0, a1, a2, a3 = symbols('a0 a1 a2 a3', real=True)
    
    local_dict = {
        'x': x,
        'a0': a0, 'a1': a1, 'a2': a2, 'a3': a3,
        'pow': Lambda((x, symbols('y')), Pow(Abs(x), symbols('y'))),
        'abs': Lambda(x, Abs(x)),
    }
    return sympify(expr_str, locals=local_dict)

# This creates a SYMBOLIC expression
expr_from_string = parse_expression("pow(abs(a0 - x), a1)")
print(f"Parsed from string: {expr_from_string}")
print(f"Type of a0: {type(expr_from_string.free_symbols)}")

## 5. Complete Workflow

Here's how to safely compute limits for density profile validation:

In [ ]:
def compute_limits_safely(expr, x, params):
    """
    Compute limits after substituting numerical parameter values.
    
    Parameters
    ----------
    expr : sympy expression (may have symbolic a0, a1, ...)
    x : sympy symbol for the variable
    params : dict like {'a0': -3.83, 'a1': -2.81}
    
    Returns
    -------
    lim_zero, lim_inf : float or None
    """
    # Create symbol -> value mapping
    subs_dict = {symbols(k, real=True): v for k, v in params.items()}
    
    # Substitute numerical values
    expr_numerical = expr.subs(subs_dict)
    print(f"After substitution: {expr_numerical}")
    
    # Now compute limits
    try:
        lim_zero = float(N(limit(expr_numerical, x, 0, '+')))
    except Exception as e:
        print(f"Error computing lim(x->0+): {e}")
        lim_zero = None
    
    try:
        lim_inf = float(N(limit(expr_numerical, x, oo)))
    except Exception as e:
        print(f"Error computing lim(x->inf): {e}")
        lim_inf = None
    
    return lim_zero, lim_inf

In [ ]:
# Test with the problematic expression
x = symbols('x', positive=True)
a0_sym, a1_sym = symbols('a0 a1', real=True)
expr = Pow(Abs(a0_sym - x), a1_sym)

print(f"Original expression: {expr}\n")

# Test with various parameter values
test_params = [
    {'a0': -3.83, 'a1': -2.81},
    {'a0': -0.99, 'a1': -1.45},
    {'a0': 1.0, 'a1': -2.0},
]

for params in test_params:
    print(f"Parameters: {params}")
    lim_zero, lim_inf = compute_limits_safely(expr, x, params)
    print(f"  lim(x->0+) = {lim_zero}")
    print(f"  lim(x->inf) = {lim_inf}")
    
    # Validate for density profile
    zero_ok = lim_zero is not None and lim_zero > 0 and np.isfinite(lim_zero)
    inf_ok = lim_inf is not None and abs(lim_inf) < 1e-10
    print(f"  Valid density profile: {zero_ok and inf_ok}\n")

## Summary

| Scenario | Abs() | limit() |
|----------|-------|--------|
| `a0 = -3.83` (Python float) | Simplified away | Fast |
| `a0 = symbols('a0')` (sympy symbol) | Kept as Abs() | Can hang |
| Symbolic + `.subs({a0: -3.83})` | Simplified away | Fast |

**Key insight**: Always substitute numerical values before computing limits!